# Introduction: Fav

In [0]:
%sh
cd ../
pip install . 
# --force-reinstall --no-deps 
pip install s3fs yfinance

In [0]:
import sys
sys.path.append('.')
from DB_MA_finetuning_qkcvlm import * # Ensure all dependencies are installed correctly
start_time = time.time()

_v = 1
v_qkcv = -1


### Dataset Creation

In [0]:
config_qkcv = Config_qkcv(_v)
config_qkcv.v_qkcv=v_qkcv

config_qkcv.split_predictions=True

_col_forecast=config_qkcv.get_col_forecast()
print(_col_forecast)

horizon = 32

### reader
import pyarrow.parquet as pq
import pandas as pd
import s3fs,os
fs = s3fs.S3FileSystem()

_p_base = 's3://'

_p_base_retail = os.path.join(_p_base, 'favorita-grocery-sales-forecasting')
file_db=f"Database_db_Favorita_{_col_forecast}_.csv"


_df_items = pd.read_csv(os.path.join(_p_base_retail,'items.csv'))
_df_stores = pd.read_csv(os.path.join(_p_base_retail,'stores.csv'))
_df_stores.rename(columns={'type': 'type_store'}, inplace=True)


### features not used in original model
_features_static = [ 'city', 'state', 'type_store', 'cluster'] + ['family', 'class', 'perishable']
_features_dynamic = ['type', 'locale', 'locale_name','transferred']

_use_staging_train = True # skip data proceeding
_save_to_staging = True # update staging file

In [0]:
### load data
import glob

if not _use_staging_train:
    _df_train = pd.read_csv(os.path.join(_p_base_retail, 'train.csv'), sep=',')

    _df_train = _df_train.loc[(_df_train.date >= '0015-01-01') & (_df_train.date < '2016-03-01')]
    gc.collect()

    print(f"date.min {_df_train.date.min()}, date.max {_df_train.date.max()}, item_nbr.max {_df_train.item_nbr.max()}, unit_sales.min {_df_train.unit_sales.min()}, unit_sales.max() {_df_train.unit_sales.max()}")

    unique_id = 'siid'
    _df_train[unique_id] = _df_train['store_nbr']*100000000 + _df_train['item_nbr']

    df_cross_join = _df_train[['date']].drop_duplicates().assign(key=1).merge(_df_train[[unique_id, 'store_nbr', 'item_nbr']].drop_duplicates().assign(key=1), on='key').drop('key', axis=1)

    _df_train['open_flag'] = 1
    _df_train = df_cross_join.merge(_df_train, on=['date', unique_id, 'store_nbr', 'item_nbr'], how='left').sort_values([unique_id, 'date']).reset_index()

    _df_train.count()
    _df_train.isnull().sum()

    _df_train['unit_sales'] = _df_train.groupby(unique_id)['unit_sales'].ffill()
    _df_train = _df_train.loc[(_df_train.date >= '2015-01-01') & (_df_train.date < '2016-03-01')]

    _df_train.isnull().sum()

    _df_train['unit_sales'].fillna(0, inplace=True)
    _df_train['open_flag'].fillna(0, inplace=True)

    _df_complete_numeric = _df_train.merge(_df_oil, on='date', how='left').merge(_df_holidays_events, on='date', how='left')

    _df_complete_numeric.rename(columns={unique_id: 'unique_id',
                                'unit_sales': 'y',
                                'date': 'ds'}, inplace=True)
    _df_complete_numeric["ds"] = pd.to_datetime(_df_complete_numeric["ds"])

    _df_train = None
    gc.collect()

    for _c in _features_dynamic:
        _df_complete_numeric[_c] = _df_complete_numeric[_c].astype('category').cat.codes
    _dt_max = _df_complete_numeric.ds.max()
    _df_complete_numeric = _df_complete_numeric.loc[_df_complete_numeric.item_nbr.isin(_df_complete_numeric.loc[_df_complete_numeric.ds == _dt_max, 'item_nbr'])].reset_index(drop=True)

    _df_complete_numeric.y.fillna(0.0001, inplace=True)

    if _save_to_staging:

        print(f"saving files, y max {_df_complete_numeric['y'].max()}, min {_df_complete_numeric['y'].min()}")
        _df_complete_numeric.ds.max()

        ### save _df_complete_numeric
        num_splits = 10

        # Split the dataframe into smaller dataframes
        df_splits = np.array_split(_df_complete_numeric, num_splits)

        # Save each split into a separate parquet file
        for i, df_split in enumerate(df_splits):
            df_split.to_parquet(os.path.join(_p_base_retail, f'_df_complete_numeric_part_{i}.parquet'))

else:
    # Read all parquet files in the specified directory
    parquet_files =[]
    for i in range(10):
        parquet_files.append(os.path.join(_p_base_retail, f'_df_complete_numeric_part_{i}.parquet'))
    print(f'parquet_files {parquet_files}')
    # Concatenate all the parquet files into a single dataframe
    _df_complete_numeric = pd.concat([pd.read_parquet(file) for file in parquet_files])

_df_complete_numeric.describe()

### _df_static
_df_static = _df_complete_numeric[['unique_id','store_nbr','item_nbr']].drop_duplicates().merge(_df_stores, on='store_nbr', how='left').merge(_df_items, on='item_nbr', how='left')

_df_static_numeric = _df_static.copy().drop(columns=['store_nbr', 'item_nbr',])
for _c in _features_static:
    _df_static_numeric[_c] = _df_static_numeric[_c].astype('category').cat.codes

Y_train_df = _df_complete_numeric.loc[(_df_complete_numeric.ds >= '2015-01-01') & (_df_complete_numeric.ds < '2015-12-01'), _features_dynamic + ['unique_id', 'ds', 'y']]

Y_test_df = _df_complete_numeric.loc[(_df_complete_numeric.ds >= '2016-01-01') & (_df_complete_numeric.ds < '2016-02-01'), _features_dynamic + ['unique_id', 'ds', 'y']]

### Model Creation

In [0]:
model, hparams, tfm_config = get_model(_features_static, 
                                       config_qkcv,
                                       horizon,
                                       load_weights=True)


In [0]:

predictions, y_future, model_tunned, finetuner, ca, attention_score = prediction_pipeline(tfm_config, horizon, config_qkcv, Y_train_df, Y_test_df, _df_static_numeric, model)



In [0]:
if not config_qkcv.train_only:
    df_merged, pred_vals_tunc = post_predictions(predictions, y_future)
    print(_v)
    print(_col_forecast)
    print(f'{_v}_Fav_{_col_forecast}')


    # Example usage
    wpe_func(df_merged, eval_horizon = [horizon],forecast='forecast')

    # Example usage
    calculate_matrix(df_merged)
    mae = calculate_mae(df_merged)
    print(f"Mean Absolute Error (MAE): {mae}")

In [0]:
print(f"Execution time: {(time.time() - start_time)/60:.2f} mins")

In [0]:
folder_path = f"{_p_base}/results"

filename = f"Favorita_{_v}_{_col_forecast}"

if finetuner!=-1:
    pd.DataFrame(finetuner.training_matrix).to_csv(f"{folder_path}/matrix_{filename}.csv", index=False)

df_merged.to_csv(f"{folder_path}/df_merged_{filename}.csv", index=False)

ca.to_csv(f"{folder_path}/ca_{filename}.csv", index=False)
attention_score.to_csv(f"{folder_path}/attention_score_{filename}.csv", index=False)

